# Inferencia HybridCVCNN — Pipeline completo

**Pipeline:**
1. Seleccionar una señal `.pt` por tipo de target y SNR
2. Lanzar el detector de entropía Shannon → detectar bursts
3. Visualizar la señal con `plot_muestra` y `plot_espectrograma_3d`
4. Construir la entrada al modelo (crop IQ centrado + 12 features físicas)
5. Clasificación binaria: **Drone / No Drone**

---
**Targets disponibles:**
- `0,1,2,3,5,6` → label=1 (drone)
- `4`           → label=0 (ruido)

**SNRs disponibles:** -20, -18, -16, ..., +30 dB (paso 2)

In [ ]:
# ── Configuración ──────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, r'c:\repos\DroneDetectionRF')
os.environ['PYTHONIOENCODING'] = 'utf-8'

# ===== PARAMETROS AJUSTABLES =====
TARGET_ID = 4        # 0,1,2,3,5,6 = drone | 4 = ruido
SNR_DB    = -18        # dB: -20, -18, ..., +30
SAMPLE_IDX = 3         # índice dentro de las señales con ese target+SNR (0 = primera encontrada)

DATA_DIR  = r'C:\TFM_data\NoisyUAV\drone_RF_data'
CKPT_PATH = r'c:\repos\DroneDetectionRF\NoisyUAV\resultados_hybrid_run2\checkpoints\best_model.pt'
CACHE_PATH= r'c:\repos\DroneDetectionRF\NoisyUAV\resultados_hybrid\features_cache.npz'
# ==================================

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import torch
import numpy as np
from pathlib import Path
import glob

from NoisyUAV.funciones.detector_entropia import (
    detectar_bursts, print_diagnostico, plot_muestra, plot_espectrograma_3d
)
from NoisyUAV.funciones.physical_features import extract_features, load_features_cache
from NoisyUAV.modelos.hybrid_cvcnn import HybridCVCNN

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── 1. Buscar y cargar señal .pt ───────────────────────────────────────────────
# Formato del nombre de fichero: IQdata_sample{N}_target{T}_snr{S}.pt
pattern = str(Path(DATA_DIR) / f'*target{TARGET_ID}_snr{SNR_DB}*.pt')
matches = sorted(glob.glob(pattern))

if not matches:
    raise FileNotFoundError(
        f'No hay señales con target={TARGET_ID}, SNR={SNR_DB} dB en {DATA_DIR}\n'
        f'Patrón buscado: {pattern}'
    )

filepath = matches[SAMPLE_IDX % len(matches)]
print(f'Fichero cargado ({SAMPLE_IDX+1}/{len(matches)} disponibles):')
print(f'  {Path(filepath).name}')

d   = torch.load(filepath, map_location='cpu', weights_only=False)
iq  = d['x_iq'].float()   # [2, 1048576]

label_real = 1 if TARGET_ID != 4 else 0
print(f'\nLabel real: {label_real}  ({"DRONE" if label_real==1 else "RUIDO"})')
print(f'Forma IQ:  {list(iq.shape)}  ({iq.shape[1]/14e6*1000:.1f} ms @ 14 MHz)')

In [ ]:
# ── 2. Detector de Entropía Shannon ───────────────────────────────────────────
FS      = 14e6
NPERSEG = 2048
Z_THRESH      = 2.5
MIN_BURST_MS  = 0.3
MERGE_GAP_MS  = 1.5
BG_MULT       = 4.0
MAX_BINS_FRAC = 0.25
ADAPTIVE_MS   = 15.0

t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq,
    fs              = FS,
    nperseg         = NPERSEG,
    z_thresh        = Z_THRESH,
    min_burst_ms    = MIN_BURST_MS,
    merge_gap_ms    = MERGE_GAP_MS,
    bg_mult         = BG_MULT,
    max_bins_frac   = MAX_BINS_FRAC,
    adaptive_window_ms = ADAPTIVE_MS,
)

# ── 3. Figura interactiva: 5 paneles (amplitud, espectrograma, PSD, H(m), bins) ──
titulo = f'Target {TARGET_ID}  |  SNR={SNR_DB} dB  |  {Path(filepath).name}'

fig = plot_muestra(
    iq, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
    fs=FS, nperseg=NPERSEG,
    z_thresh=Z_THRESH, bg_mult=BG_MULT,
    max_bins_frac=MAX_BINS_FRAC,
    adaptive_window_ms=ADAPTIVE_MS,
    titulo=titulo
)
fig.show()

# Diagnóstico textual
print_diagnostico(
    t_ms, nf_v, ns, umbral_v, n_active, bursts,
    nperseg=NPERSEG, fs=FS,
    z_thresh=Z_THRESH, bg_mult=BG_MULT,
    max_bins_frac=MAX_BINS_FRAC,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    target=TARGET_ID, snr=SNR_DB
)

In [ ]:
# # ── 4. Espectrograma 3D interactivo ───────────────────────────────────────────
# fig3d = plot_espectrograma_3d(
#     iq, fs=FS, nperseg=NPERSEG,
#     smooth_sigma=2.0,
#     titulo=titulo
# )
# fig3d.show()

## Prueba modelo normal

In [ ]:
# ── 5. Cargar modelo HybridCVCNN ──────────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
cfg  = ckpt.get('cfg', {})

model = HybridCVCNN(
    phys_dim     = int(cfg.get('phys_dim',    12)),
    cnn_embed    = int(cfg.get('cnn_embed',   256)),
    pool_size    = int(cfg.get('pool_size',   32)),
    hidden_dim   = int(cfg.get('hidden_dim',  256)),
    dropout_cnn  = 0.0,
    dropout_fuse = 0.0,
).to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

phys_mean = torch.tensor(ckpt['phys_mean'], dtype=torch.float32)
phys_std  = torch.tensor(ckpt['phys_std'],  dtype=torch.float32)
crop_len  = int(cfg.get('crop_len', 131072))

print(f'Modelo cargado  |  Best epoch: {ckpt["epoch"]}  |  Val F1: {ckpt["val_f1"]:.4f}')
print(f'crop_len = {crop_len} muestras = {crop_len/FS*1000:.1f} ms')

In [ ]:
# ── 6. Construir entradas al modelo ───────────────────────────────────────────
#
# DISEÑO ACTUAL: 
#   - Rama CNN:     crop CENTRAL de la señal completa
#   - Rama física:  12 features de la señal COMPLETA via detector de entropía
#
# NOTA DE DISEÑO (ver memory.md Sección 3.3):
#   El crop puede NO coincidir con el burst detectado. El modelo fue entrenado
#   así y funciona con 84.79% de accuracy a nivel de fichero.
#   Una mejora futura (Burst-Guided Crop) centraría el crop en el burst.

# --- Rama CNN: crop central ---
L = iq.shape[1]
if L <= crop_len:
    pad = torch.zeros(2, crop_len - L)
    iq_crop = torch.cat([iq, pad], dim=1)
else:
    start   = (L - crop_len) // 2
    iq_crop = iq[:, start:start + crop_len]

# Normalización RMS (OBLIGATORIA — igual que en entrenamiento)
power   = iq_crop.pow(2).mean().clamp(min=1e-12).sqrt()
iq_crop = iq_crop / power
print(f'IQ crop: {list(iq_crop.shape)}  |  RMS post-norm: {iq_crop.pow(2).mean().sqrt():.4f}')

# --- Rama física: 12 features del FICHERO COMPLETO ---
# Intenta usar caché; si no, las computa en tiempo real
filename = Path(filepath).name
try:
    cache = load_features_cache(CACHE_PATH)
    if filename in cache:
        feats_raw = torch.from_numpy(cache[filename].copy())
        print(f'Features cargadas desde caché.')
    else:
        raise KeyError('no en caché')
except Exception:
    feats_raw = torch.from_numpy(extract_features(iq, fs=FS, nperseg=NPERSEG))
    print('Features computadas en tiempo real (no estaban en caché).')

# Z-score normalización (mismas estadísticas que en entrenamiento)
feats_norm = (feats_raw - phys_mean) / (phys_std + 1e-8)
feats_norm = torch.clamp(feats_norm, -5.0, 5.0)

print(f'\n12 Features físicas (valores brutos):')
names = ['noise_floor','noise_sigma','mean_n_active','p75_n_active',
         'H_min','H_mean','n_bursts','dur_ms_main','z_peak_main','drop_b_main','n_act_main','dur_total']
for n, v in zip(names, feats_raw.tolist()):
    print(f'  {n:20s}: {v:.4f}')

In [ ]:
# ── 7. Inferencia ─────────────────────────────────────────────────────────────
with torch.no_grad():
    iq_in   = iq_crop.unsqueeze(0).to(DEVICE)       # [1, 2, crop_len]
    feat_in = feats_norm.unsqueeze(0).to(DEVICE)     # [1, 12]
    logit   = model(iq_in, feat_in)                  # [1, 1]
    prob    = torch.sigmoid(logit).item()
    pred    = int(prob > 0.5)

print('=' * 60)
print(f'  RESULTADO DE CLASIFICACION')
print('=' * 60)
print(f'  Probabilidad de drone : {prob:.4f}  ({prob*100:.1f}%)')
print(f'  Prediccion            : {"DRONE" if pred == 1 else "RUIDO"}  (umbral 0.5)')
print(f'  Label real            : {"DRONE" if label_real == 1 else "RUIDO"}')
print()
if pred == label_real:
    print(f'  >>> CORRECTO ✓')
else:
    print(f'  >>> INCORRECTO ✗  (falso {"positivo" if pred==1 else "negativo"})')
print('=' * 60)
print()
print(f'  Detecciones del detector de entropía:')
if bursts:
    for k, b in enumerate(bursts):
        print(f'    Burst {k+1}: {b["t0"]:.1f}-{b["t1"]:.1f} ms  '
              f'dur={b["dur_ms"]:.2f}ms  z={b["z_peak"]:.1f}  bins={b["n_act"]:.0f}')
else:
    print(f'    Ningún burst detectado (señal por debajo del umbral CFAR)')
print()
print(f'  Crop CNN: muestras [{(L-crop_len)//2}, {(L-crop_len)//2 + crop_len}]'
      f' = [{(L-crop_len)/2/FS*1e3:.1f}, {((L-crop_len)/2+crop_len)/FS*1e3:.1f}] ms')

In [ ]:
# ── 8. [OPCIONAL] Sweep de umbrales de decisión ───────────────────────────────
# ¿Con qué umbral el modelo cambiaría su decisión?
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
print('Sensibilidad al umbral de decisión:')
print(f'  Prob(drone) = {prob:.4f}')
for th in thresholds:
    dec = 'DRONE' if prob > th else 'RUIDO '
    ok  = '✓' if (int(prob > th) == label_real) else '✗'
    print(f'  umbral={th:.1f}  →  {dec}  {ok}')

In [ ]:
# ── 9. [OPCIONAL] Probar múltiples señales con el mismo target/SNR ─────────────
print(f'Evaluando todas las {len(matches)} señales disponibles (target={TARGET_ID}, SNR={SNR_DB}dB)...')
results = []

with torch.no_grad():
    for fp in matches:
        d_   = torch.load(fp, map_location='cpu', weights_only=False)
        iq_  = d_['x_iq'].float()
        L_   = iq_.shape[1]

        # Crop central
        if L_ <= crop_len:
            pad = torch.zeros(2, crop_len - L_)
            iq_c = torch.cat([iq_, pad], dim=1)
        else:
            s = (L_ - crop_len) // 2
            iq_c = iq_[:, s:s + crop_len]
        pwr  = iq_c.pow(2).mean().clamp(min=1e-12).sqrt()
        iq_c = iq_c / pwr

        # Features
        key_ = Path(fp).name
        try:
            fr = torch.from_numpy(cache[key_].copy())
        except Exception:
            fr = torch.from_numpy(extract_features(iq_, fs=FS, nperseg=NPERSEG))
        fn = torch.clamp((fr - phys_mean) / (phys_std + 1e-8), -5., 5.)

        logit_ = model(iq_c.unsqueeze(0).to(DEVICE), fn.unsqueeze(0).to(DEVICE))
        p_     = torch.sigmoid(logit_).item()
        results.append({'file': key_, 'prob': p_, 'pred': int(p_ > 0.5), 'label': label_real})

acc = sum(r['pred'] == r['label'] for r in results) / len(results)
prob_mean = np.mean([r['prob'] for r in results])
print(f'\nAccuracy: {acc:.3f} ({acc*100:.1f}%)  |  Prob(drone) media: {prob_mean:.3f}')
print(f'Predicciones: {sum(r["pred"]==1 for r in results)} DRONE, {sum(r["pred"]==0 for r in results)} RUIDO de {len(results)} señales')

# Prueba inferencia específica por burst

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
cfg  = ckpt.get('cfg', {})
model = HybridCVCNN(
    phys_dim     = int(cfg.get('phys_dim',    12)),
    cnn_embed    = int(cfg.get('cnn_embed',   256)),
    pool_size    = int(cfg.get('pool_size',   32)),
    hidden_dim   = int(cfg.get('hidden_dim',  256)),
    dropout_cnn  = 0.0,
    dropout_fuse = 0.0,
).to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
phys_mean = torch.tensor(ckpt['phys_mean'], dtype=torch.float32)
phys_std  = torch.tensor(ckpt['phys_std'],  dtype=torch.float32)
print(f'Modelo antiguo cargado  |  Época óptima: {ckpt["epoch"]}  |  Val F1: {ckpt["val_f1"]:.4f}')
# ── 5. EXPERIMENTO: Evaluación Burst a Burst Local (Una sola señal) ──────────
print(f'\n--- EXPERIMENTO LOCAL: Evaluando únicamente la señal actual seleccionada ---')
print(f'Fichero: {Path(filepath).name}')
probabilidades_bursts = []

In [ ]:
with torch.no_grad():
    if len(bursts) == 0: 
        print(f"  El detector CFAR no encontró NADA en esta señal. Decidimos RUIDO por defecto (0.0%)")
        prob_global = 0.0
    else:
        for k, b in enumerate(bursts):
            start_idx = max(0, int((b['t0'] / 1000.0) * FS))
            end_idx   = min(iq.shape[1], int((b['t1'] / 1000.0) * FS))
            
            if start_idx >= end_idx:
                continue
                
            # Recorte exacto
            iq_crop = iq[:, start_idx:end_idx]
            
            # Normalización OBLIGATORIA al recorte microscópico
            pwr  = iq_crop.pow(2).mean().clamp(min=1e-12).sqrt()
            iq_crop = iq_crop / pwr
            
            # Features Locales de este recorte + Estandarización vieja
            try:
                fr_local = torch.from_numpy(extract_features(iq_crop, fs=FS, nperseg=NPERSEG))
            except Exception:
                fr_local = torch.zeros(12) 
                
            fn_local = torch.clamp((fr_local - phys_mean) / (phys_std + 1e-8), -5., 5.)
            
            # Inferencia: Pasamos el trocito a la antigua CNN (soporta variable len)
            logit_b = model(iq_crop.unsqueeze(0).to(DEVICE), fn_local.unsqueeze(0).to(DEVICE))
            prob_b = torch.sigmoid(logit_b).item()
            probabilidades_bursts.append(prob_b)
            
            print(f"  Burst {k+1:02d} | Intervalo: {b['t0']:5.1f}-{b['t1']:5.1f}ms | Probabilidad Local (DRON): {prob_b*100:5.1f}%")
        # Agrupación estadística final: Si AL MENOS UN burst de la sala parece dron, decrétalo
        if len(probabilidades_bursts) > 0:
            prob_global = max(probabilidades_bursts)
        else:
            prob_global = 0.0

In [ ]:
print('\n' + '=' * 60)
print(f'  RESULTADO FINAL TRAS ENSAMBLAR TODOS LOS BURSTS LOCALES')
print('=' * 60)
print(f'  Probabilidad global ensamblada: {prob_global:.4f}  ({prob_global*100:.1f}%)')
prediccion = "DRONE" if prob_global > 0.5 else "RUIDO"
real_text  = "DRONE" if label_real == 1 else "RUIDO"
print(f'  La Red predice ahora que es   : {prediccion}')
print(f'  La Etiqueta original es       : {real_text}')
print(f'  >>> {"CORRECTO ✓" if (prob_global > 0.5) == label_real else "INCORRECTO ✗  (El Cambio de Dominio falló)"}')